In [3]:
from pathlib import Path
import json
from docling.document_converter import DocumentConverter

pdf_path = Path("/home/enma/Projects/Med360_RAG_Bot/data/pdf/cardiology_all.pdf")
output_dir = Path("/home/enma/Projects/Med360_RAG_Bot/data/processed")
output_dir.mkdir(parents=True, exist_ok=True)

stem = pdf_path.stem
master_json_path = output_dir / f"{stem}.docling.json"
markdown_path = output_dir / f"{stem}.md"

converter = DocumentConverter()
result = converter.convert(str(pdf_path))
doc = result.document

with open(master_json_path, "w", encoding="utf-8") as f:
    json.dump(doc.export_to_dict(), f, ensure_ascii=False, indent=2)

with open(markdown_path, "w", encoding="utf-8") as f:
    f.write(doc.export_to_markdown())

print(master_json_path)
print(markdown_path)

[INFO] 2026-03-10 15:12:38,827 [RapidOCR] base.py:22: Using engine_name: torch
[INFO] 2026-03-10 15:12:38,828 [RapidOCR] device_config.py:64: Using GPU device with ID: 0
[INFO] 2026-03-10 15:12:38,853 [RapidOCR] download_file.py:60: File exists and is valid: /home/enma/miniconda3/envs/medrag_all/lib/python3.10/site-packages/rapidocr/models/ch_PP-OCRv4_det_infer.pth
[INFO] 2026-03-10 15:12:38,854 [RapidOCR] main.py:50: Using /home/enma/miniconda3/envs/medrag_all/lib/python3.10/site-packages/rapidocr/models/ch_PP-OCRv4_det_infer.pth
[INFO] 2026-03-10 15:12:39,170 [RapidOCR] base.py:22: Using engine_name: torch
[INFO] 2026-03-10 15:12:39,171 [RapidOCR] device_config.py:64: Using GPU device with ID: 0
[INFO] 2026-03-10 15:12:39,175 [RapidOCR] download_file.py:60: File exists and is valid: /home/enma/miniconda3/envs/medrag_all/lib/python3.10/site-packages/rapidocr/models/ch_ptocr_mobile_v2.0_cls_infer.pth
[INFO] 2026-03-10 15:12:39,175 [RapidOCR] main.py:50: Using /home/enma/miniconda3/envs

/home/enma/Projects/Med360_RAG_Bot/data/processed/cardiology_all.docling.json
/home/enma/Projects/Med360_RAG_Bot/data/processed/cardiology_all.md


In [4]:
#!/usr/bin/env python3
"""
02_clean_md.py

Purpose:
- Clean raw markdown from Docling
- Normalize headings, bullets, spacing
- Remove obvious extraction noise
- Preserve placeholders for images/tables for later repair
- Add source metadata block for RAG

Usage:
python 02_clean_md.py \
  --raw_md_dir /home/enma/Projects/Med360_RAG_Bot/data/extracted/raw_md \
  --output_dir /home/enma/Projects/Med360_RAG_Bot/data/processed/clean_md
"""

from __future__ import annotations

import argparse
import re
from pathlib import Path
from datetime import datetime


def now_iso() -> str:
    return datetime.utcnow().isoformat() + "Z"


def read_text(path: Path) -> str:
    return path.read_text(encoding="utf-8", errors="ignore")


def write_text(path: Path, text: str) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(text, encoding="utf-8")


def normalize_newlines(text: str) -> str:
    return text.replace("\r\n", "\n").replace("\r", "\n")


def collapse_spaces(text: str) -> str:
    lines = []
    for line in text.split("\n"):
        line = re.sub(r"[ \t]+", " ", line).strip()
        lines.append(line)
    return "\n".join(lines)


def remove_redundant_blank_lines(text: str) -> str:
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip() + "\n"


def normalize_bullets(text: str) -> str:
    lines = []
    for line in text.split("\n"):
        if re.match(r"^[•·▪◦]\s*", line):
            line = re.sub(r"^[•·▪◦]\s*", "- ", line)
        lines.append(line)
    return "\n".join(lines)


def normalize_headings(text: str) -> str:
    """
    Heuristic:
    - If a short line is standalone and followed by text, promote it to heading
    - Avoid changing lines that already start with markdown heading markers
    """
    lines = text.split("\n")
    out = []

    for i, line in enumerate(lines):
        stripped = line.strip()

        if not stripped:
            out.append("")
            continue

        if stripped.startswith("#"):
            out.append(stripped)
            continue

        prev_blank = i == 0 or not lines[i - 1].strip()
        next_exists = i + 1 < len(lines)
        next_nonempty = next_exists and bool(lines[i + 1].strip())

        looks_like_heading = (
            prev_blank
            and next_nonempty
            and len(stripped) <= 80
            and not stripped.endswith(".")
            and stripped.lower() == stripped.lower()
            and len(stripped.split()) <= 10
            and not re.match(r"^[-*]\s+", stripped)
            and "<!-- image -->" not in stripped
        )

        if looks_like_heading:
            out.append(f"## {stripped}")
        else:
            out.append(stripped)

    return "\n".join(out)


def fix_hyphenated_linebreaks(text: str) -> str:
    text = re.sub(r"(\w)-\n(\w)", r"\1\2", text)
    return text


def join_wrapped_lines(text: str) -> str:
    """
    Joins lines that are likely just wrapped paragraph text,
    but preserves headings, bullets, tables, and image placeholders.
    """
    lines = text.split("\n")
    merged = []
    buffer = []

    def flush_buffer():
        nonlocal buffer, merged
        if buffer:
            merged.append(" ".join(buffer).strip())
            buffer = []

    for line in lines:
        stripped = line.strip()

        special = (
            not stripped
            or stripped.startswith("#")
            or stripped.startswith("- ")
            or stripped.startswith("|")
            or stripped == "<!-- image -->"
            or re.match(r"^\d+\.\s+", stripped)
        )

        if special:
            flush_buffer()
            merged.append(stripped)
        else:
            if buffer and re.search(r"[.:;!?)]$", buffer[-1]):
                flush_buffer()
            buffer.append(stripped)

    flush_buffer()
    return "\n".join(merged)


def remove_duplicate_lines(text: str) -> str:
    lines = text.split("\n")
    out = []
    prev = None
    for line in lines:
        if line == prev and line.strip():
            continue
        out.append(line)
        prev = line
    return "\n".join(out)


def normalize_image_placeholders(text: str) -> str:
    count = 0
    lines = []
    for line in text.split("\n"):
        if "<!-- image -->" in line:
            count += 1
            lines.append(f"## Figure Placeholder {count}\n[IMAGE_PLACEHOLDER_{count}]")
        else:
            lines.append(line)
    return "\n".join(lines)


def add_metadata_block(text: str, source_file: str, doc_type: str = "medical_pdf") -> str:
    title = None
    for line in text.split("\n"):
        stripped = line.strip()
        if stripped.startswith("# "):
            title = stripped[2:].strip()
            break

    if title is None:
        title = Path(source_file).stem.replace("_", " ").title()
        text = f"# {title}\n\n{text}"

    meta = (
        f"## Source Metadata\n"
        f"- source_file: {Path(source_file).name}\n"
        f"- source_type: {doc_type}\n"
        f"- cleaned_on: {now_iso()}\n\n"
    )

    if "## Source Metadata" not in text:
        parts = text.split("\n", 2)
        if len(parts) >= 2 and parts[0].startswith("# "):
            text = parts[0] + "\n\n" + meta + ("\n".join(parts[1:]).lstrip())
        else:
            text = meta + text

    return text


def clean_markdown(raw_text: str, source_file: str) -> str:
    text = raw_text
    text = normalize_newlines(text)
    text = fix_hyphenated_linebreaks(text)
    text = collapse_spaces(text)
    text = normalize_bullets(text)
    text = join_wrapped_lines(text)
    text = normalize_headings(text)
    text = normalize_image_placeholders(text)
    text = remove_duplicate_lines(text)
    text = remove_redundant_blank_lines(text)
    text = add_metadata_block(text, source_file=source_file)
    text = remove_redundant_blank_lines(text)
    return text


def process_file(md_path: Path, output_dir: Path) -> None:
    raw_text = read_text(md_path)
    source_file = md_path.name.replace(".raw.md", ".pdf")
    cleaned = clean_markdown(raw_text, source_file=source_file)

    out_path = output_dir / md_path.name.replace(".raw.md", ".clean.md")
    write_text(out_path, cleaned)
    print(f"[OK] {md_path.name}")


def main() -> None:
    parser = argparse.ArgumentParser()
    parser.add_argument("--raw_md_dir", type=str, required=True)
    parser.add_argument("--output_dir", type=str, required=True)
    args = parser.parse_args()

    raw_md_dir = Path(args.raw_md_dir)
    output_dir = Path(args.output_dir)

    files = sorted(raw_md_dir.glob("*.md"))
    if not files:
        print("No markdown files found.")
        return

    for md_path in files:
        process_file(md_path, output_dir)


if __name__ == "__main__":
    main()

usage: ipykernel_launcher.py [-h] --raw_md_dir RAW_MD_DIR --output_dir
                             OUTPUT_DIR
ipykernel_launcher.py: error: the following arguments are required: --raw_md_dir, --output_dir


SystemExit: 2

/home/enma/miniconda3/envs/medrag_all/lib/python3.10/site-packages/IPython/core/interactiveshell.py:3587: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
